<a href="https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/janaadarwish/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1) My lane and why

**Provisional lane:** Content refresh prioritization (page-level), using the starter dataset.

**Why this lane:** When I ran notebook 01, a learned model picked the pages worth reviewing first about three times better than a hand-written rule (Precision@50 of roughly 0.24 vs roughly 0.74). That gap shows there is real signal in the data. I want to understand which signals drive it and whether a team could act on them.

I can confirm or change this lane until the end of Week 4.

## 2) The question

- **Search question:** Which pages should a content team review or refresh first to recover or improve search performance?
- **Unit of analysis:** One page.
- **Output:** A ranked review queue (a score per page), not a yes/no verdict.
- **Decision:** Which pages get limited review time this cycle.
- **Action:** An SEO or content team reviews and refreshes the top-ranked pages (for example, the top 50).
- **Cost of a wrong call:**
  - False positive: review time is spent on a page that didn't need it.
  - False negative: a page that needed attention is missed and keeps underperforming.
  - Both costs are modest and recoverable, because a human reviews each flagged page before anything changes.
- **Why data or ML can help:** Nobody can read tens of thousands of pages. A simple rule can only use one or two signals at a time, while a model can combine many signals and rank pages consistently. It is only worth using if it beats a simple baseline, which I will test.
- **Why this is not just "train a model":** The point is a decision under a limited review budget. The model is useful only if its ranking is better than the rule a person would use anyway, and if the output is readable enough to act on.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [15]:
import os, glob
import pandas as pd

if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/janaadarwish/flyrank-ml-internship.git
%cd flyrank-ml-internship

print(glob.glob("data/**/*.csv", recursive=True))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 164 (delta 59), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.86 MiB | 12.47 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
['data/raw/content_refresh_anonymized.csv']


In [16]:
import os
if not os.path.exists("flyrank-ml-internship"):
    !git clone https://github.com/janaadarwish/flyrank-ml-internship.git
%cd flyrank-ml-internship
print(os.listdir("."))
print(os.listdir("data"))

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 164 (delta 59), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 1.86 MiB | 8.19 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
['data', 'skills', 'LICENSE', '.gitignore', 'SETUP.md', 'outputs', 'DATA_USE.md', 'requirements.txt', 'docs', 'work', 'CLAUDE.md', 'scripts', 'notebooks', 'README.md', 'assets', '.github', 'submission', 'GUIDE.md', '.git', 'AGENTS.md', 'index.html']
['raw']


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [17]:
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)
print(list(df.columns))

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [18]:
print("Pages:", df["content_id"].nunique(), "| Clients:", df["client_id"].nunique())
print(df["trend_direction"].value_counts(normalize=True).round(3))
print(df["freshness_tier"].value_counts(normalize=True).round(3))
print("Median days since last update:", df["days_since_last_update"].median())

Pages: 30000 | Clients: 32
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64
freshness_tier
0-30      0.683
91-180    0.306
31-90     0.006
181+      0.006
Name: proportion, dtype: float64
Median days since last update: 20.0


## 3) Quick look at the data

- The starter dataset has **30,000 pages** from **32 clients**, with 44 columns.
- **54.2%** of pages have a `trend_direction` of "down", against 14.6% "up", 19.9% "stable", 7.5% "new" and 3.8% "flat". This suggests many pages show declining search performance, so a team has more candidates than it can review.
- Pages were updated recently: **68.3%** are in the 0-30 day freshness tier, the median is **20 days** since the last update, and almost none fall in the 31-90 or 181+ tiers. Because most pages are recent, "last updated" alone is unlikely to separate pages well, and I'll check it against other signals.

These numbers describe the anonymized starter sample. They suggest a refresh-scoring lane is worth pursuing because there are many candidates to rank, but they don't show how the full dataset behaves.

## 4) Careful words: what I can and can't claim

**I can say:**
- These results are observed on an anonymized starter sample.
- The model ranks pages better than a simple rule in this sample (directional evidence).
- The ranking is decision-support for a human reviewer.

**I can't say:**
- That the model predicts how Google ranks pages.
- That any feature causes a ranking change.
- That the results hold on the full dataset or for other sites without testing.

## 5) Self-check

- [x] Picked a lane and explained why
- [x] Named the decision and the action
- [x] Showed at least two real numbers from the starter data
- [x] Explained why this is more than "train a model"
- [x] Used careful language (observed / directional / decision-support)